In [1]:
import os
import json
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import random
import math

In [2]:
os.chdir('..')

In [3]:
rmse_frame = pd.read_csv('./data/error_frame.csv')
proba_frame = pd.read_csv('./data/proba_summary_frame.csv')

In [4]:
with open('./data/names.json') as file:
    names = json.load(file)

In [5]:
season_rmse = rmse_frame[~rmse_frame['playoffs']]
season_proba = proba_frame[~proba_frame['playoffs']]

playoff_rmse = rmse_frame[rmse_frame['playoffs']]
playoff_proba = proba_frame[proba_frame['playoffs']]

In [6]:
final_cols = ['matchupid', 'week', 'home_guid', 'away_guid', 'playoffs', 'true_score','winner_guid', 'loser_guid', '_c_k_60_osa_40_proba', '_c_k_60_osa_20_proba', 'default_proba']

elo_configs = ['_c_k_60_osa_40_', '_c_k_60_osa_20_', 'default_']

In [7]:
default_error = season_rmse.default_error
k60osa20_error = season_rmse._c_k_60_osa_20_error
k60osa40_error = season_rmse._c_k_60_osa_40_error

In [8]:
default_vig = default_error.mean() + 2 * default_error.std()
k60osa20_vig = k60osa20_error.mean() + 2 * k60osa20_error.std()
k60osa40_vig = k60osa40_error.mean() + 2 * k60osa40_error.std()

vigs = (k60osa40_vig, k60osa20_vig, default_vig)
print(default_error.mean())
print(default_error.std())
print(default_vig)
print(k60osa20_vig)
print(k60osa40_vig)

0.025440706109578794
0.03720865074886493
0.09985800760730866
0.10069196685326584
0.102245230917217


In [9]:
gamble_frame = season_proba.rename(columns={'Unnamed: 0': 'matchupid'})[final_cols].reset_index(drop=True)
gamble_frame['season'] = gamble_frame.matchupid.astype(str).str[:4].astype(int)
gamble_frame['true_spread'] = (gamble_frame['true_score'] - .5) * 9

In [10]:
for i, col in enumerate(elo_configs):
    amer_odds = np.zeros(gamble_frame.shape[0])
    spr = col + 'spread'
    amer = col + 'amer'
    gamble_frame[spr] = round(18 * ((1 - vigs[i]) * (.5 - gamble_frame[col + 'proba'])))/2
    odds = gamble_frame[col + 'proba'].values.copy()
    odds[gamble_frame[col + 'proba'] < .5] = 1 - gamble_frame[col + 'proba'][gamble_frame[col + 'proba'] < .5]
    gamble_frame[amer] = 10000 * odds * (1 + vigs[i])/(odds * 100 - 100)

In [11]:
for i, col in enumerate(elo_configs):
    matchups = gamble_frame.shape[0]
    spr = col + 'spread'
    amer = col + 'amer'
    cov = col + 'coverage'
    mon = col + 'money_coverage'
    fav = col + 'fav_coverage'
    prob = col + 'proba'
    
    spread_res = gamble_frame['true_spread'] + gamble_frame[spr] 
    coverage = pd.Series(['push'] * matchups)
    coverage[spread_res > 0] = gamble_frame['winner_guid'][spread_res > 0]
    coverage[spread_res < 0] = gamble_frame['loser_guid'][spread_res < 0]
    coverage[coverage == 'tie'] = 'push'
    gamble_frame[cov] = coverage
    
    winner = ((gamble_frame['winner_guid'] == gamble_frame[cov]).astype(float) - .5) * 2
    winner[winner > 0] -= vigs[i]
    gamble_frame[col + 'spr_winnings'] = winner
    
    money_cover = pd.Series(['push'] * matchups)
    true_score = gamble_frame.true_score
    money_cover[true_score > 0.5] = gamble_frame['home_guid'][true_score > 0.5]
    money_cover[true_score < 0.5] = gamble_frame['away_guid'][true_score < 0.5]
    gamble_frame[mon] = money_cover
    
    mon_win = pd.Series([-1] * matchups)
    fav_win = pd.Series([False] * matchups)
    fav_win[(gamble_frame.true_score < .5) & (gamble_frame[prob] < 0.5)] = True
    fav_win[(gamble_frame.true_score > .5) & (gamble_frame[prob] > 0.5)] = True
    mon_win[fav_win] = -100.0/gamble_frame[amer]
    mon_win[gamble_frame[prob] == 0.5] = 0
    gamble_frame[col + 'money_line_winnings'] = mon_win
    
    fav_covs = pd.Series(['loss'] * matchups)
    fav_covs[mon_win > 0] = gamble_frame[mon][mon_win > 0]
    gamble_frame[fav] = fav_covs

In [12]:
spr_coverage = pd.Series(Counter(coverage[abs(gamble_frame.default_spread) >= 1.5])).astype(float) # the last one run is default, which is what we want to analyze
print(spr_coverage)
mon_coverage = pd.Series(Counter(money_cover)).astype(float)
print(mon_coverage)
fav_coverage = pd.Series(Counter(fav_covs)).astype(float)
print(fav_coverage)
total_matchups = pd.Series(Counter(gamble_frame.home_guid) + Counter(gamble_frame.away_guid))
spread_matchups = pd.Series(Counter(gamble_frame.home_guid[abs(gamble_frame.default_spread) >= 1.5]) + Counter(gamble_frame.away_guid[abs(gamble_frame.default_spread) >= 1.5]))

SZTYHNWXEJSWLLFCUGMIRQJSJM    1.0
QOSB6TNJH6AZSUXHZXGSCF6KGY    3.0
RVVFXOWC3I4X4EX5WSDEA4OQYI    4.0
LNRT67BZLFW4H3EUP2TDHOG74M    3.0
YWVPE4432SD4KI54TYIBYSUY4E    3.0
BNYD3F2FQK3NFXVMAJW4RPJKOA    1.0
VOKZZSOK4ZQXCRZS4X3C6A2UKM    2.0
NVAFN332AJMUVGCGC3CVBDWNQQ    4.0
NIHSTHKSFAGS6M5GEP36QG4T64    5.0
VWB4LBHYWJRAQ2E66SHEWFTCRI    2.0
VFTMFPVMSCHKRTIILW7BM5UIGA    2.0
Q4COTZKZODDIVSW3XXCJH7RNLE    1.0
CCYS62GDCHMWDZZZQDTB35DTRE    3.0
dtype: float64
NIHSTHKSFAGS6M5GEP36QG4T64     55.0
VOKZZSOK4ZQXCRZS4X3C6A2UKM     81.0
LNRT67BZLFW4H3EUP2TDHOG74M    107.0
RVVFXOWC3I4X4EX5WSDEA4OQYI     31.0
QOSB6TNJH6AZSUXHZXGSCF6KGY     87.0
5NJCJX76R73J5ONZN2HXCZUSYU     13.0
YPMRGP6D6AZCDFMT7LTZQ5NKL4      6.0
MOCKVC7YTGWQWP2GG2DA6R4MTM      1.0
push                           23.0
K4ON2S6OGMFWP26Y7K54HITJDU      6.0
VWB4LBHYWJRAQ2E66SHEWFTCRI     80.0
DuckHead                        3.0
SZTYHNWXEJSWLLFCUGMIRQJSJM     64.0
YWVPE4432SD4KI54TYIBYSUY4E     62.0
BNYD3F2FQK3NFXVMAJW4RPJKOA     43.0
AM6

In [13]:
rate_df = pd.concat([total_matchups, spread_matchups, mon_coverage, fav_coverage, spr_coverage], axis=1).fillna(0).astype(float)
rate_df.columns = ['matchups', 'spread_matchups', 'money_covers', 'favored_covers', 'spread_covers']


In [14]:
rate_df.matchups['push'] = sum(rate_df.matchups)/2
rate_df.spread_matchups['push'] = sum(rate_df.spread_matchups)/2
rate_df['money_rate'] = (rate_df.money_covers/rate_df.matchups * 100).round(2).astype(str) + '%'
rate_df['favored_win_rate'] = (rate_df.favored_covers/rate_df.matchups * 100).round(2).astype(str) + '%'
rate_df['spread_rate'] = (rate_df.spread_covers/rate_df.spread_matchups * 100).round(2).astype(str) + '%'

In [15]:
rate_df.rename(index=names).sort_values('spread_matchups', ascending=False)

,matchups,spread_matchups,money_covers,favored_covers,spread_covers,money_rate,favored_win_rate,spread_rate
push,808.0,34.0,23.0,0.0,0.0,2.85%,0.0%,0.0%
Alison,158.0,8.0,55.0,24.0,5.0,34.81%,15.19%,62.5%
Andrew E,81.0,7.0,31.0,21.0,4.0,38.27%,25.93%,57.14%
Chris,158.0,6.0,81.0,52.0,2.0,51.27%,32.91%,33.33%
John,158.0,6.0,107.0,77.0,3.0,67.72%,48.73%,50.0%
James W,21.0,6.0,1.0,0.0,3.0,4.76%,0.0%,50.0%
Ravi,136.0,6.0,80.0,42.0,2.0,58.82%,30.88%,33.33%
Rohan,158.0,5.0,87.0,49.0,3.0,55.06%,31.01%,60.0%
Alex K,77.0,5.0,25.0,13.0,4.0,32.47%,16.88%,80.0%
Neil,116.0,4.0,62.0,37.0,3.0,53.45%,31.9%,75.0%


In [16]:
gamble_frame.groupby(['season']).agg('sum')['default_money_line_winnings']

/tmp/ipykernel_11172/2902339425.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  gamble_frame.groupby(['season']).agg('sum')['default_money_line_winnings']


season
2014    25.983158
2015    -6.666657
2016    30.116038
2017    18.798338
2018     6.606559
2019     2.342416
2020   -10.638931
2021     6.384458
Name: default_money_line_winnings, dtype: float64

In [17]:
gamble_frame.groupby(['week']).agg('sum')['default_money_line_winnings']

/tmp/ipykernel_11172/3692009859.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  gamble_frame.groupby(['week']).agg('sum')['default_money_line_winnings']


week
0     0.000000
1    -2.359698
2     3.167530
3    -2.801372
4     1.877160
5     3.530292
6     5.586558
7     6.050026
8    -1.541305
9     1.308583
10    5.707147
11    6.402080
12    9.850182
13    1.263648
14    6.282189
15    7.990825
16    6.993219
17    0.798085
18    7.572274
19    3.134748
20    2.651723
21   -0.538516
Name: default_money_line_winnings, dtype: float64

In [18]:
gamble_frame[abs(gamble_frame.default_spread) >= 1.5].groupby(['season']).agg('sum')['default_spr_winnings']

/tmp/ipykernel_11172/1887047804.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  gamble_frame[abs(gamble_frame.default_spread) >= 1.5].groupby(['season']).agg('sum')['default_spr_winnings']


season
2015    0.900142
2017    1.400852
2018   -2.000000
2019   -4.399432
2021   -3.299574
Name: default_spr_winnings, dtype: float64

In [19]:
gamble_frame[abs(gamble_frame.default_spread) >= 1.5].groupby(['week']).agg('sum')['default_spr_winnings']

/tmp/ipykernel_11172/1021606018.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  gamble_frame[abs(gamble_frame.default_spread) >= 1.5].groupby(['week']).agg('sum')['default_spr_winnings']


week
7    -1.000000
10   -0.099858
11   -2.000000
12   -0.099858
13   -0.199716
14    3.600568
15   -2.099858
16   -3.000000
17    0.800284
18   -4.099858
19    0.800284
Name: default_spr_winnings, dtype: float64

In [40]:
g = gamble_frame.replace({'default_fav_coverage': names}).groupby(['season', 'default_fav_coverage']).agg('sum')['default_money_line_winnings']

/tmp/ipykernel_11172/3165286944.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  g = gamble_frame.replace({'default_fav_coverage': names}).groupby(['season', 'default_fav_coverage']).agg('sum')['default_money_line_winnings']


In [41]:
print(g.reset_index().to_string())

    season default_fav_coverage  default_money_line_winnings
0     2014               Alison                     5.274167
1     2014             Andrew E                     4.056863
2     2014                Chris                     9.474059
3     2014              Chris S                     2.207053
4     2014                 John                    10.644700
5     2014                Rohan                     9.517785
6     2014              Sandeep                     6.808532
7     2014                 loss                   -22.000000
8     2015               Alison                     2.260878
9     2015             Andrew E                     4.792432
10    2015               Austin                     1.513706
11    2015                Chris                     9.286019
12    2015             DuckHead                     1.773498
13    2015              Guillem                     6.169887
14    2015                 John                     3.674492
15    2015              

In [22]:
print((gamble_frame.default_money_line_winnings > 0).sum())
print(gamble_frame.groupby(['season']).agg('sum')['default_money_line_winnings'].mean())
print(gamble_frame.default_money_line_winnings.sum())
print(gamble_frame.default_spr_winnings.sum())

480
9.115672365613154
72.92537892490523
-9.940363195069656


/tmp/ipykernel_11172/986153702.py:2: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  print(gamble_frame.groupby(['season']).agg('sum')['default_money_line_winnings'].mean())


In [23]:
pdefault_error = playoff_rmse.default_error
pk60osa20_error = playoff_rmse._c_k_60_osa_20_error
pk60osa40_error = playoff_rmse._c_k_60_osa_40_error

In [24]:
pdefault_vig = pdefault_error.mean() + 2 * pdefault_error.std()
pk60osa20_vig = pk60osa20_error.mean() + 2 * pk60osa20_error.std()
pk60osa40_vig = pk60osa40_error.mean() + 2 * pk60osa40_error.std()

pvigs = (pk60osa40_vig, pk60osa20_vig, pdefault_vig)

print(pdefault_error.mean())
print(pdefault_error.std())
print(pdefault_vig)
print(pk60osa20_vig)
print(pk60osa40_vig)

0.03474081268175911
0.048398751200836344
0.1315383150834318
0.1311513173900141
0.13079297012414937


In [36]:
inj = 1
funds = 1
for i in gamble_frame.default_money_line_winnings:
    funds += i
    if funds < 1:
        funds +=1
        inj += 1
print(inj)

2


In [25]:
pgamble_frame = playoff_proba.rename(columns={'Unnamed: 0': 'matchupid'})[final_cols].reset_index(drop=True)
pgamble_frame['season'] = pgamble_frame.matchupid.astype(str).str[:4].astype(int)
pgamble_frame['true_spread'] = (pgamble_frame['true_score'] - .5) * 9

In [26]:
for i, col in enumerate(elo_configs):
    amer_odds = np.zeros(pgamble_frame.shape[0])
    spr = col + 'spread'
    amer = col + 'amer'
    pgamble_frame[spr] = round(18 * ((1 - vigs[i]) * (.5 - pgamble_frame[col + 'proba'])))/2
    odds = pgamble_frame[col + 'proba'].values.copy()
    odds[pgamble_frame[col + 'proba'] < .5] = 1 - pgamble_frame[col + 'proba'][pgamble_frame[col + 'proba'] < .5]
    pgamble_frame[amer] = 10000 * odds * (1 + pvigs[i])/(odds * 100 - 100)

In [27]:
for i, col in enumerate(elo_configs):
    matchups = pgamble_frame.shape[0]
    spr = col + 'spread'
    amer = col + 'amer'
    cov = col + 'coverage'
    mon = col + 'money_coverage'
    prob = col + 'proba'
    
    pspread_res = pgamble_frame['true_spread'] + pgamble_frame[spr] 
    pcoverage = pd.Series(['push'] * matchups)
    pcoverage[pspread_res > 0] = pgamble_frame['winner_guid'][pspread_res > 0]
    pcoverage[pspread_res < 0] = pgamble_frame['loser_guid'][pspread_res < 0]
    pcoverage[coverage == 'tie'] = 'push'
    pgamble_frame[cov] = pcoverage
    
    pwinner = ((pgamble_frame['winner_guid'] == pgamble_frame[cov]).astype(float) - .5) * 2
    pwinner[pwinner > 0] -= pvigs[i]
    pgamble_frame[col + 'spr_winnings'] = pwinner
    
    pmoney_cover = pd.Series(['push'] * matchups)
    ptrue_score = pgamble_frame.true_score
    pmoney_cover[ptrue_score > 0.5] = pgamble_frame['home_guid'][ptrue_score > 0.5]
    pmoney_cover[ptrue_score < 0.5] = pgamble_frame['away_guid'][ptrue_score < 0.5]
    pgamble_frame[mon] = pmoney_cover
    
    pmon_win = pd.Series([-1] * matchups)
    pfav_win = pd.Series([False] * matchups)
    pfav_win[(pgamble_frame.true_score < .5) & (pgamble_frame[prob] < 0.5)] = True
    pfav_win[(pgamble_frame.true_score > .5) & (pgamble_frame[prob] > 0.5)] = True
    pmon_win[pfav_win] = -100.0/pgamble_frame[amer]
    pmon_win[pgamble_frame[prob] == 0.5] = 0
    pgamble_frame[col + 'money_line_winnings'] = pmon_win
    
    pfav_covs = pd.Series(['loss'] * matchups)
    pfav_covs[pmon_win > 0] = pgamble_frame[mon][pmon_win > 0]
    pgamble_frame[fav] = pfav_covs

In [28]:
Counter(pcoverage)

Counter({'NIHSTHKSFAGS6M5GEP36QG4T64': 8,
         'VOKZZSOK4ZQXCRZS4X3C6A2UKM': 9,
         'RVVFXOWC3I4X4EX5WSDEA4OQYI': 4,
         'MOCKVC7YTGWQWP2GG2DA6R4MTM': 1,
         'YPMRGP6D6AZCDFMT7LTZQ5NKL4': 1,
         'QOSB6TNJH6AZSUXHZXGSCF6KGY': 12,
         'VWB4LBHYWJRAQ2E66SHEWFTCRI': 4,
         'LNRT67BZLFW4H3EUP2TDHOG74M': 7,
         'DuckHead': 1,
         'YWVPE4432SD4KI54TYIBYSUY4E': 6,
         'SZTYHNWXEJSWLLFCUGMIRQJSJM': 4,
         'AM6VVTB2K73KCUXPIP6ZNO5ERU': 1,
         'push': 3,
         'BNYD3F2FQK3NFXVMAJW4RPJKOA': 4,
         'VFTMFPVMSCHKRTIILW7BM5UIGA': 4,
         'PMEMCDE2SSHJ5YKGBVJZ4PI62E': 2,
         'NVAFN332AJMUVGCGC3CVBDWNQQ': 2,
         '5RADOHUTBULBXBWUKY4J275JZE': 1})

In [29]:
Counter(pmoney_cover)

Counter({'NIHSTHKSFAGS6M5GEP36QG4T64': 8,
         'VOKZZSOK4ZQXCRZS4X3C6A2UKM': 6,
         'RVVFXOWC3I4X4EX5WSDEA4OQYI': 3,
         '5NJCJX76R73J5ONZN2HXCZUSYU': 2,
         'YPMRGP6D6AZCDFMT7LTZQ5NKL4': 1,
         'QOSB6TNJH6AZSUXHZXGSCF6KGY': 7,
         'SZTYHNWXEJSWLLFCUGMIRQJSJM': 4,
         'LNRT67BZLFW4H3EUP2TDHOG74M': 8,
         'BNYD3F2FQK3NFXVMAJW4RPJKOA': 3,
         'YWVPE4432SD4KI54TYIBYSUY4E': 8,
         'VWB4LBHYWJRAQ2E66SHEWFTCRI': 8,
         'push': 3,
         'VFTMFPVMSCHKRTIILW7BM5UIGA': 6,
         'NVAFN332AJMUVGCGC3CVBDWNQQ': 3,
         'PMEMCDE2SSHJ5YKGBVJZ4PI62E': 2,
         'NZ2X6OUUEORPJ3ZFHCL4EBEU6J': 1,
         '5RADOHUTBULBXBWUKY4J275JZE': 1})

In [30]:
pspr_coverage = pd.Series(Counter(pcoverage[abs(pgamble_frame.default_spread) >= 1.5])).astype(float) # the last one run is default, which is what we want to analyze
print(pspr_coverage)
pmon_coverage = pd.Series(Counter(pmoney_cover)).astype(float)
print(pmon_coverage)
pfav_coverage = pd.Series(Counter(pfav_covs)).astype(float)
print(pfav_coverage)
ptotal_matchups = pd.Series(Counter(pgamble_frame.home_guid) + Counter(pgamble_frame.away_guid))
pspread_matchups = pd.Series(Counter(pgamble_frame.home_guid[abs(pgamble_frame.default_spread) >= 1.5]) + Counter(pgamble_frame.away_guid[abs(pgamble_frame.default_spread) >= 1.5]))

NIHSTHKSFAGS6M5GEP36QG4T64    1.0
dtype: float64
NIHSTHKSFAGS6M5GEP36QG4T64    8.0
VOKZZSOK4ZQXCRZS4X3C6A2UKM    6.0
RVVFXOWC3I4X4EX5WSDEA4OQYI    3.0
5NJCJX76R73J5ONZN2HXCZUSYU    2.0
YPMRGP6D6AZCDFMT7LTZQ5NKL4    1.0
QOSB6TNJH6AZSUXHZXGSCF6KGY    7.0
SZTYHNWXEJSWLLFCUGMIRQJSJM    4.0
LNRT67BZLFW4H3EUP2TDHOG74M    8.0
BNYD3F2FQK3NFXVMAJW4RPJKOA    3.0
YWVPE4432SD4KI54TYIBYSUY4E    8.0
VWB4LBHYWJRAQ2E66SHEWFTCRI    8.0
push                          3.0
VFTMFPVMSCHKRTIILW7BM5UIGA    6.0
NVAFN332AJMUVGCGC3CVBDWNQQ    3.0
PMEMCDE2SSHJ5YKGBVJZ4PI62E    2.0
NZ2X6OUUEORPJ3ZFHCL4EBEU6J    1.0
5RADOHUTBULBXBWUKY4J275JZE    1.0
dtype: float64
loss                          27.0
RVVFXOWC3I4X4EX5WSDEA4OQYI     2.0
5NJCJX76R73J5ONZN2HXCZUSYU     2.0
NIHSTHKSFAGS6M5GEP36QG4T64     6.0
YPMRGP6D6AZCDFMT7LTZQ5NKL4     1.0
VOKZZSOK4ZQXCRZS4X3C6A2UKM     3.0
SZTYHNWXEJSWLLFCUGMIRQJSJM     2.0
LNRT67BZLFW4H3EUP2TDHOG74M     6.0
QOSB6TNJH6AZSUXHZXGSCF6KGY     3.0
BNYD3F2FQK3NFXVMAJW4RPJKOA     2.0
YWVPE443

In [31]:
prate_df = pd.concat([ptotal_matchups, pspread_matchups, pmon_coverage, pfav_coverage, pspr_coverage], axis=1).fillna(0).astype(float)
prate_df.columns = ['matchups', 'spread_matchups', 'money_covers', 'favored_covers', 'spread_covers']

In [32]:
prate_df.matchups['push'] = sum(prate_df.matchups)/2
prate_df.spread_matchups['push'] = sum(prate_df.spread_matchups)/2
prate_df['money_rate'] = (prate_df.money_covers/prate_df.matchups * 100).round(2).astype(str) + '%'
prate_df['favored_win_rate'] = (prate_df.favored_covers/prate_df.matchups * 100).round(2).astype(str) + '%'
prate_df['spread_rate'] = (prate_df.spread_covers/prate_df.spread_matchups * 100).round(2).astype(str) + '%'

In [33]:
prate_df.rename(index=names).sort_values('money_rate', ascending=False)

,matchups,spread_matchups,money_covers,favored_covers,spread_covers,money_rate,favored_win_rate,spread_rate
loss,0.0,0.0,0.0,27.0,0.0,nan%,inf%,nan%
Neil,12.0,0.0,8.0,7.0,0.0,66.67%,58.33%,nan%
Sahil,9.0,0.0,6.0,5.0,0.0,66.67%,55.56%,nan%
Ravi,14.0,0.0,8.0,4.0,0.0,57.14%,28.57%,nan%
Alison,15.0,1.0,8.0,6.0,1.0,53.33%,40.0%,100.0%
John,15.0,0.0,8.0,6.0,0.0,53.33%,40.0%,nan%
Guillem,8.0,0.0,4.0,2.0,0.0,50.0%,25.0%,nan%
Aaron,2.0,0.0,1.0,0.0,0.0,50.0%,0.0%,nan%
Alex K,6.0,0.0,3.0,3.0,0.0,50.0%,50.0%,nan%
Chris S,2.0,0.0,1.0,1.0,0.0,50.0%,50.0%,nan%


In [34]:
pgamble_frame.groupby(['season']).agg('sum')['default_money_line_winnings']

/tmp/ipykernel_11172/2739383770.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  pgamble_frame.groupby(['season']).agg('sum')['default_money_line_winnings']


season
2014    0.487117
2015    2.005987
2016    5.152498
2017   -4.315303
2018   -0.661952
2020    4.490792
2021   -0.379316
Name: default_money_line_winnings, dtype: float64

In [35]:
pgamble_frame[abs(pgamble_frame.default_spread) >= 1.5].groupby(['season']).agg('sum')['default_spr_winnings']

/tmp/ipykernel_11172/3279597916.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  pgamble_frame[abs(pgamble_frame.default_spread) >= 1.5].groupby(['season']).agg('sum')['default_spr_winnings']


season
2021    0.868462
Name: default_spr_winnings, dtype: float64

In [42]:
print((pgamble_frame.default_money_line_winnings > 0).sum())
print(pgamble_frame.groupby(['season']).agg('sum')['default_money_line_winnings'].mean())
print(pgamble_frame.default_money_line_winnings.sum())
print(pgamble_frame.default_spr_winnings.sum())

47
0.9685458579535414
6.7798210056747905
4.475390766495863


/tmp/ipykernel_11172/1500236374.py:2: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  print(pgamble_frame.groupby(['season']).agg('sum')['default_money_line_winnings'].mean())


In [37]:
(gamble_frame._c_k_60_osa_20_money_line_winnings > 0).mean()

0.620049504950495

In [43]:
inj = 1
funds = 1
for i in pgamble_frame.default_money_line_winnings:
    funds += i
    if funds < 1:
        funds +=1
        inj += 1
print(inj)

3
